In [1]:
import numpy as np
import random

In [2]:
class Value:
    def __init__(self, value, children = (), op = '', label = '', grad = 0):
        
        self.value = value
        self._prev = children
        self._backward = lambda : None
        self.grad = grad
        self._op = op
        self.label = label

    def __repr__(self):
        
        return f'Value <data = {self.value}>'

    def __add__(self, other):

        if isinstance(other, Value):
            new_val = other.value

            out = Value(new_val + self.value, (self, other), '+')
            
            def _backward():
                self.grad += 1.0 * out.grad
                other.grad += 1.0 * out.grad 
            out._backward = _backward
            
        else:
            return NotImplemented
        
        return out

    def __mul__(self, other):
        
        if isinstance(other, Value):
            new_val = other.value

            out = Value(new_val * self.value, (self, other), '*')
            
            def _backward():
                self.grad += out.grad * other.value
                other.grad += out.grad * self.value
            out._backward = _backward
            
        else:
            return NotImplemented
        
        return out

    def tanh(self):
        x = self.value
        t = (np.exp(2*x) - 1) / (np.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += out.grad * (1.0 - t**2)
        out._backward = _backward
        
        return out

    def backward(self):
        
        topo = []
        visited = set()
        
        def dfs(node):
    
            if node in visited:
                return
    
            visited.add(node)
    
            for child in node._prev:
                dfs(child)
    
            topo.append(node)
    
        dfs(self)
    
        self.grad = 1

        for node in reversed(topo):
            node._backward()

In [3]:
class Neuron:

    def __init__(self, n):
        self.w = [Value(random.uniform(-1,1)) for _ in range(n)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        aggr = sum((xi*wi for xi, wi in zip(x, self.w)), self.b)
        activated = aggr.tanh()
        return aggr

    def __repr__(self):
        return f'Neuron<Weights: {self.w}, Bias: {self.b}>'

In [4]:
class Layer:

    def __init__(self, inp, out):
        self.inp = inp
        self.out = out
        self.neurons = [Neuron(self.inp) for _ in range(self.out)]

    def __call__(self, x):
        out = [neuron(x) for neuron in self.neurons]
        return out

In [9]:
layer = Layer(3,3)

In [10]:
x = [Value(i) for i in range(1,4)]
x

[Value <data = 1>, Value <data = 2>, Value <data = 3>]

In [11]:
layer.neurons

[Neuron<Weights: [Value <data = 0.2731649647187102>, Value <data = 0.36675360335248186>, Value <data = 0.2987486030956421>], Bias: Value <data = 0.6702025735953614>>,
 Neuron<Weights: [Value <data = 0.9369015800522968>, Value <data = 0.9887116660827981>, Value <data = 0.017305809171517916>], Bias: Value <data = -0.07772150861022187>>,
 Neuron<Weights: [Value <data = 0.271307844431661>, Value <data = 0.4140031383380127>, Value <data = -0.5844599929321845>], Bias: Value <data = -0.14722552084189>>]

In [12]:
output = layer(x)
output

[Value <data = 2.5731205543059614>,
 Value <data = 2.888520831122225>,
 Value <data = -0.8012913785307572>]